In [1]:
import torch, os
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

import kagglehub
path = kagglehub.dataset_download("bryanpark/sudoku")
print(path, os.listdir(path))

True Tesla T4


100%|██████████| 68.1M/68.1M [00:04<00:00, 16.5MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/bryanpark/sudoku/versions/3 ['sudoku.csv']


In [2]:
import numpy as np, pandas as pd

csv_path = os.path.join(path, "sudoku.csv")
df = pd.read_csv(csv_path, dtype=str)          # no nrows -> whole file
print(df.shape)                                 # (1000000, 2)

def to_arr(series):
    b = "".join(series.tolist()).encode()
    return (np.frombuffer(b, dtype=np.uint8) - 48).reshape(-1, 9, 9)   # uint8, 81 MB

X = to_arr(df["quizzes"])
Y = to_arr(df["solutions"])
del df
print(X.shape, Y.shape)

(1000000, 2)
(1000000, 9, 9) (1000000, 9, 9)


In [3]:
import torch.nn as nn, torch.nn.functional as F
device = "cuda"

Xt = torch.from_numpy(X).to(device)             # uint8
Yt = torch.from_numpy(Y).to(device)

n_val = 10_000
Xtr, Y_tr = Xt[:-n_val], Yt[:-n_val]
Xva, Yva  = Xt[-n_val:], Yt[-n_val:]

In [4]:
def make_batch(xq, ys):
    xq, ys = xq.long(), ys.long()
    blank = (xq == 0)
    frac = torch.rand(xq.size(0), 1, 1, device=device) * 0.8
    reveal = (torch.rand(xq.shape, device=device) < frac) & blank
    board = torch.where(reveal, ys, xq)
    inp = F.one_hot(board, 10).permute(0, 3, 1, 2).float()
    target = ys - 1
    mask = (board == 0)
    return inp, target, mask

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, padding=1); self.b1 = nn.BatchNorm2d(c)
        self.c2 = nn.Conv2d(c, c, 3, padding=1); self.b2 = nn.BatchNorm2d(c)
    def forward(self, x):
        h = F.relu(self.b1(self.c1(x)))
        h = self.b2(self.c2(h))
        return F.relu(x + h)

class SudokuNet(nn.Module):
    def __init__(self, c=128, n_blocks=8):
        super().__init__()
        self.inp = nn.Conv2d(10, c, 3, padding=1)
        self.blocks = nn.Sequential(*[Block(c) for _ in range(n_blocks)])
        self.out = nn.Conv2d(c, 9, 1)
    def forward(self, x):
        return self.out(self.blocks(F.relu(self.inp(x))))

def masked_loss_acc(logits, target, mask):
    loss = F.cross_entropy(logits.float(), target, reduction="none")
    n = mask.sum().clamp(min=1)
    loss = (loss * mask).sum() / n
    acc = ((logits.argmax(1) == target) & mask).sum() / n
    return loss, acc

model = SudokuNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

In [5]:
import time
scaler = torch.amp.GradScaler("cuda")
bs, epochs = 512, 5

for ep in range(epochs):
    model.train()
    perm = torch.randperm(len(Xtr), device=device)
    t0 = time.time()
    for step, i in enumerate(range(0, len(Xtr), bs)):
        idx = perm[i:i+bs]
        inp, tgt, mask = make_batch(Xtr[idx], Y_tr[idx])
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(inp)
        loss, acc = masked_loss_acc(logits, tgt, mask)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        if step % 200 == 0:
            print(f"ep {ep+1} step {step} loss {loss.item():.3f} acc {acc.item():.3f} ({time.time()-t0:.0f}s)")

    model.eval()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        inp, tgt, mask = make_batch(Xva[:5000], Yva[:5000])
        vloss, vacc = masked_loss_acc(model(inp), tgt, mask)
    print(f"epoch {ep+1}: val loss {vloss.item():.3f} | val cell acc {vacc.item():.3f}")
    torch.save(model.state_dict(), "sudoku_net.pt")   # save every epoch in case Colab disconnects

ep 1 step 0 loss 3.114 acc 0.112 (3s)
ep 1 step 200 loss 0.579 acc 0.762 (12s)
ep 1 step 400 loss 0.344 acc 0.863 (22s)
ep 1 step 600 loss 0.222 acc 0.913 (32s)
ep 1 step 800 loss 0.152 acc 0.944 (42s)
ep 1 step 1000 loss 0.126 acc 0.954 (52s)
ep 1 step 1200 loss 0.081 acc 0.971 (62s)
ep 1 step 1400 loss 0.086 acc 0.968 (72s)
ep 1 step 1600 loss 0.078 acc 0.972 (83s)
ep 1 step 1800 loss 0.070 acc 0.976 (93s)
epoch 1: val loss 0.082 | val cell acc 0.971
ep 2 step 0 loss 0.063 acc 0.977 (0s)
ep 2 step 200 loss 0.067 acc 0.976 (10s)
ep 2 step 400 loss 0.057 acc 0.979 (20s)
ep 2 step 600 loss 0.052 acc 0.981 (30s)
ep 2 step 800 loss 0.049 acc 0.982 (40s)
ep 2 step 1000 loss 0.044 acc 0.984 (50s)
ep 2 step 1200 loss 0.053 acc 0.980 (60s)
ep 2 step 1400 loss 0.044 acc 0.985 (70s)
ep 2 step 1600 loss 0.037 acc 0.988 (80s)
ep 2 step 1800 loss 0.041 acc 0.985 (90s)
epoch 2: val loss 0.043 | val cell acc 0.985
ep 3 step 0 loss 0.041 acc 0.985 (0s)
ep 3 step 200 loss 0.045 acc 0.984 (10s)
ep 3 st

In [6]:
@torch.no_grad()
def solve_batch(quizzes):                       # quizzes: (B, 9, 9) on device
    model.eval()
    board = quizzes.long().clone()
    B = board.size(0)
    ar = torch.arange(B, device=device)
    for _ in range(81):
        blank = (board == 0)
        has = blank.view(B, -1).any(1)          # puzzles that still have blanks
        if not has.any():
            break
        inp = F.one_hot(board, 10).permute(0, 3, 1, 2).float()
        probs = model(inp).softmax(1)           # (B, 9, 9, 9)
        conf, digit = probs.max(1)              # confidence + best digit per cell
        conf = conf.masked_fill(~blank, -1)     # ignore already-filled cells
        flat = conf.view(B, -1).argmax(1)       # most confident blank cell per puzzle
        r, c = flat // 9, flat % 9
        d = digit[ar, r, c] + 1                 # class 0-8 -> digit 1-9
        board[ar[has], r[has], c[has]] = d[has]
    return board

In [7]:
pred = solve_batch(Xva[:2000])
solved = (pred == Yva[:2000].long()).view(2000, -1).all(1)
print("fully solved:", solved.float().mean().item())
print("cell accuracy:", (pred == Yva[:2000].long()).float().mean().item())

fully solved: 0.9205000400543213
cell accuracy: 0.9974691867828369


In [8]:
def show(b):
    for i, row in enumerate(b.tolist()):
        if i % 3 == 0 and i: print("------+-------+------")
        print(" | ".join(" ".join(str(x) if x else "." for x in row[j:j+3]) for j in (0, 3, 6)))

i = 0
print("Puzzle:");   show(Xva[i].long().cpu())
print("\nModel:");  show(pred[i].cpu())
print("\nTruth:");  show(Yva[i].long().cpu())

Puzzle:
. . 7 | 8 . 3 | . . .
. . 1 | . 6 . | . . 2
8 3 . | . 5 . | . . 4
------+-------+------
. 2 . | . 7 . | 9 . 5
5 . . | . . . | 1 2 .
. . . | . 9 . | . 3 .
------+-------+------
1 . 5 | 4 . 2 | . . 8
. 9 . | 6 . . | . . .
. 7 3 | 5 8 . | 4 6 1

Model:
6 4 7 | 8 2 3 | 5 1 9
9 5 1 | 7 6 4 | 3 8 2
8 3 2 | 9 5 1 | 6 7 4
------+-------+------
3 2 6 | 1 7 8 | 9 4 5
5 8 9 | 3 4 6 | 1 2 7
7 1 4 | 2 9 5 | 8 3 6
------+-------+------
1 6 5 | 4 3 2 | 7 9 8
4 9 8 | 6 1 7 | 2 5 3
2 7 3 | 5 8 9 | 4 6 1

Truth:
6 4 7 | 8 2 3 | 5 1 9
9 5 1 | 7 6 4 | 3 8 2
8 3 2 | 9 5 1 | 6 7 4
------+-------+------
3 2 6 | 1 7 8 | 9 4 5
5 8 9 | 3 4 6 | 1 2 7
7 1 4 | 2 9 5 | 8 3 6
------+-------+------
1 6 5 | 4 3 2 | 7 9 8
4 9 8 | 6 1 7 | 2 5 3
2 7 3 | 5 8 9 | 4 6 1


In [10]:
import numpy as np, time

def allowed(b, r, c, d):
    if (b[r] == d).any() or (b[:, c] == d).any():
        return False
    br, bc = 3 * (r // 3), 3 * (c // 3)
    return not (b[br:br+3, bc:bc+3] == d).any()

def no_dead_cells(b):
    # False if some blank cell has no legal digit left
    for r in range(9):
        for c in range(9):
            if b[r, c] == 0:
                used = set(b[r]) | set(b[:, c]) | set(b[3*(r//3):3*(r//3)+3, 3*(c//3):3*(c//3)+3].ravel())
                if len(used - {0}) == 9:
                    return False
    return True

@torch.no_grad()
def guided(b, budget):
    if budget[0] <= 0:
        return None
    blank = (b == 0)
    if not blank.any():
        return b
    inp = F.one_hot(torch.from_numpy(b).to(device), 10).permute(2, 0, 1).float()[None]
    probs = model(inp)[0].softmax(0).cpu().numpy()          # (9, 9, 9), one transfer
    conf = np.where(blank, probs.max(0), -1)
    r, c = divmod(int(conf.argmax()), 9)                    # most confident blank cell
    for d in np.argsort(-probs[:, r, c])[:3] + 1:           # its top 3 digits
        if not allowed(b, r, c, d):
            continue
        nb = b.copy(); nb[r, c] = d
        if not no_dead_cells(nb):
            continue
        budget[0] -= 1
        res = guided(nb, budget)
        if res is not None:
            return res
    return None

In [12]:
model.eval()
fail_idx = (~solved).nonzero().flatten().tolist()
t0 = time.time()
for i in fail_idx[:10]:
    q = Xva[i].long().cpu().numpy()
    res = guided(q, budget=[60])
    ok = res is not None and (res == Yva[i].long().cpu().numpy()).all()
    print(i, "fixed" if ok else "still failed", f"{time.time()-t0:.1f}s")

5 fixed 0.2s
10 fixed 0.4s
17 fixed 0.7s
26 fixed 0.9s
46 fixed 1.1s
58 fixed 1.4s
70 still failed 1.6s
75 fixed 1.8s
77 fixed 1.9s
86 fixed 2.1s


In [13]:
fixed = 0
t0 = time.time()
for n, i in enumerate(fail_idx):
    q = Xva[i].long().cpu().numpy()
    res = guided(q, budget=[60])
    if res is not None and (res == Yva[i].long().cpu().numpy()).all():
        fixed += 1
    if n % 20 == 0:
        print(f"{n}/{len(fail_idx)}  fixed so far: {fixed}  ({time.time()-t0:.0f}s)")

print(f"failed before: {len(fail_idx)}, fixed now: {fixed}")
print("new full-solve rate:", (2000 - len(fail_idx) + fixed) / 2000)

0/159  fixed so far: 1  (0s)
20/159  fixed so far: 20  (3s)
40/159  fixed so far: 39  (7s)
60/159  fixed so far: 55  (11s)
80/159  fixed so far: 73  (14s)
100/159  fixed so far: 90  (17s)
120/159  fixed so far: 110  (21s)
140/159  fixed so far: 130  (24s)
failed before: 159, fixed now: 147
new full-solve rate: 0.994
